# 결측치·이상치 분석 준비 (`03_Missing_Outlier`)

이 노트북은 25일 전체 코호트 통합 정본에서 **모델 대상만 추출하고 데이터 계약을 검증**한다.
현재 구현 범위는 추출·검증까지이며, 결측치 처리와 이상치 분석은 다음 작업에서 진행한다.

## 1. 환경 설정과 통합 정본 로드

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'CSV_files').exists():
    PROJECT_ROOT = (PROJECT_ROOT / '../..').resolve()

DATA_PATH = PROJECT_ROOT / 'CSV_files' / '통합 버전' / 'landmark25_all_cohorts.csv'
KEYS = ['code_module', 'code_presentation', 'id_student']

assert DATA_PATH.exists(), f'통합 정본을 찾을 수 없음: {DATA_PATH}'
all_cohorts = pd.read_csv(DATA_PATH)
print('통합 정본:', DATA_PATH)
print('전체 shape:', all_cohorts.shape)

## 2. 모델 대상 추출

25일 시점 재학 상태를 확인할 수 있는 `eligible_at_25 == 1`이면서
`cohort_status_25 == 'model_eligible'`인 행만 선택한다. 원본 통합 CSV는 변경하지 않는다.

In [ ]:
model_df = all_cohorts.loc[
    (all_cohorts['eligible_at_25'] == 1)
    & (all_cohorts['cohort_status_25'] == 'model_eligible')
].copy()

model_df.reset_index(drop=True, inplace=True)
model_df.shape

## 3. 데이터 계약 검증

행 수, 복합 키 유일성, 타깃 결측·값 범위와 양성 건수를 확인한다.

In [ ]:
assert len(all_cohorts) == 32_593, '전체 코호트 행 수 불일치'
assert all_cohorts.duplicated(KEYS).sum() == 0, '전체 코호트 키 중복 발생'
assert len(model_df) == 27_661, '모델 대상 행 수 불일치'
assert model_df.duplicated(KEYS).sum() == 0, '모델 대상 키 중복 발생'
assert model_df['target_churn_after_25'].notna().all(), '모델 대상 타깃 결측 발생'
assert set(model_df['target_churn_after_25'].unique()) <= {0, 1}, '타깃이 0/1 이외의 값을 포함'
assert model_df['target_churn_after_25'].sum() == 5_247, '양성 건수 불일치'
assert model_df['eligible_at_25'].eq(1).all(), '제외 코호트 혼입'
assert model_df['cohort_status_25'].eq('model_eligible').all(), '코호트 상태 불일치'

validation_summary = pd.Series({
    '전체 코호트 행 수': len(all_cohorts),
    '모델 대상 행 수': len(model_df),
    '복합 키 중복': model_df.duplicated(KEYS).sum(),
    '타깃 결측': model_df['target_churn_after_25'].isna().sum(),
    '양성 건수': int(model_df['target_churn_after_25'].sum()),
    '음성 건수': int((model_df['target_churn_after_25'] == 0).sum()),
})
validation_summary

## 다음 작업

`model_df`를 기준으로 결측치의 원인을 일반 결측과 구조적 결측으로 구분한다.
이 노트북에서는 아직 대치, 행 제거, 이상치 변환을 수행하지 않았다.